In [1]:
from datasets import load_dataset
import torch as t
from transformers import AutoModelForCausalLM, AutoTokenizer
import sys
sys.path.append("..")
#from sparsify.sparsify import Sae
from huggingface_hub import hf_hub_download
from safetensors import safe_open
import json
from types import SimpleNamespace # Import SimpleNamespace
from sae_lens import SAE
# Manually load SAE due to state_dict key mismatch
repo_id = "fnlp/Llama-Scope-R1-Distill"
file_name = "400M-Slimpajama-400M-OpenR1-Math-220k/L15R"
device = "cuda"

# # Download necessary files
# config_path = hf_hub_download(repo_id=repo_id, filename=f"{file_name}/config.json")
# safetensors_path = hf_hub_download(repo_id=repo_id, filename=f"{file_name}/sae_weights.safetensors")

# # Load config
# with open(config_path, 'r') as f:
#     sae_cfg = json.load(f)
#     print(sae_cfg)

# # Convert dict to SimpleNamespace for attribute access
# sae_cfg['device'] = device
# sae_cfg['num_latents'] = 32768
# sae_cfg['transcode'] = False
# sae_cfg['normalize_decoder'] = False
# sae_cfg['skip_connection'] = False
# sae_cfg["shuffle_seed"] = 42
# sae_cfg["k"] = 50
# sae_cfg["activation"] = "topk"
# # sae_cfg["dead_feature_threshold"] = 10000000
# sae_cfg["multi_topk"] = False
# sae_cfg["threshold"] = 0.0
# sae_cfg_obj = SimpleNamespace(**sae_cfg)
# sae = Sae(cfg=sae_cfg_obj, d_in=4096)


SAE_LAYER = 15
RELEASE = "llama_scope_r1_distill"
SAE_ID = f"l{SAE_LAYER}r_400m_slimpajama_400m_openr1_math"
DEVICE = "cuda:3" if t.cuda.is_available() else "cpu"

sae, cfg_dict, sparsity = SAE.from_pretrained(
    # see other options in sae_lens/pretrained_saes.yaml
    release=RELEASE,
    sae_id=SAE_ID,
    device=DEVICE
)

sae.eval()

SAE(
  (activation_fn): ReLU()
  (hook_sae_input): HookPoint()
  (hook_sae_acts_pre): HookPoint()
  (hook_sae_acts_post): HookPoint()
  (hook_sae_output): HookPoint()
  (hook_sae_recons): HookPoint()
  (hook_sae_error): HookPoint()
)

In [2]:
from autointerp.vis.dashboard import make_feature_display

cache_path = "/share/u/koyena/llama-8b-cache-sae-lens/model.layers.15"

features = list(range(100))
feature_display = make_feature_display([cache_path], features)

Output(layout=Layout(border_bottom='1px solid #ddd', border_left='1px solid #ddd', border_right='1px solid #dd…

In [3]:
# check what tokens are in the cache
cache_path = "/share/u/koyena/llama-8b-cache-sae-lens/model.layers.15"
cache = t.load(f"{cache_path}/0.pt")
print(cache)
# see all the tokens in the cache
print(cache['activations'].shape)


{'locations': tensor([[   0,    4,  311],
        [   0,    4,  356],
        [   0,    5,  257],
        ...,
        [1999,  510,  191],
        [1999,  511,  191],
        [1999,  511,  387]]), 'activations': tensor([1.6696, 1.5928, 1.5744,  ..., 2.1070, 2.2615, 3.5380]), 'tokens_path': '/share/u/koyena/llama-8b-cache-sae-lens/tokens.pt', 'model_id': 'deepseek-ai/DeepSeek-R1-Distill-Llama-8B'}
torch.Size([33936250])


In [3]:

from autointerp.vis.dashboard import make_feature_display
import os # Add os import if not already there

cache_path = "/disk/u/koyena/llama-8b-cache-sae-lens-with-slimpajama/model.layers.15"

features_to_load = [17587]
# Assuming the hookpoint name is the last part of the cache_path directory
hookpoint_name = os.path.basename(cache_path)
features_dict = {hookpoint_name: features_to_load}

# Also add the ctx_len argument back, as it was needed before
feature_display = make_feature_display([cache_path], features_dict, ctx_len=1024)

Output(layout=Layout(border_bottom='1px solid #ddd', border_left='1px solid #ddd', border_right='1px solid #dd…

DEBUG: Dictionary passed to display (from make_feature_display): {'model.layers.15': [Feature(index=17587, max_activation=15.350531578063965, activating_examples=[Example(tokens=tensor([128000, 128011,   5618,  ...,  15636,     11,    369]), str_tokens=['<｜begin▁of▁sentence｜>', '<｜User｜>', 'Please', ' reason', ' step', ' by', ' step', ',', ' and', ' put', ' your', ' final', ' answer', ' within', ' \\', 'boxed', '{}.', 'Let', ' $', 'S', '_{', 'n', '}', '=\\', '{', '1', ',n', ',n', '^{', '2', '},', 'n', '^{', '3', '},', ' \\', 'cd', 'ots', ' \\', '}$', ',', ' where', ' $', 'n', '$', ' is', ' an', ' integer', ' greater', ' than', ' $', '1', '$.', ' Find', ' the', ' smallest', ' number', ' $', 'k', '=k', '(n', ')$', ' such', ' that', ' there', ' is', ' a', ' number', ' which', ' may', ' be', ' expressed', ' as', ' a', ' sum', ' of', ' $', 'k', '$', ' (', 'possibly', ' repeated', ')', ' elements', ' in', ' $', 'S', '_{', 'n', '}$', ' in', ' more', ' than', ' one', ' way', '.', ' (', 'R', 'e

In [2]:
from autointerp.vis.dashboard import make_dashboard
sae.to("cuda:3")
cache_path = "/disk/u/koyena/llama-8b-cache-sae-lens-with-slimpajama/model.layers.15"
dashboard = make_dashboard(cache_path, sae.encode, in_memory=False,  ctx_len=1024)

Loading checkpoint shards:   0%|          | 0/2 [00:00<?, ?it/s]

In [4]:
import pandas as pd
df = pd.read_parquet("/share/u/koyena/llama-8b-cache-three/model.layers.15/header.parquet")
print(df.info())
print(f'\nFeature 22219 exists: {22219 in df['feature_idx'].values}')

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 32299 entries, 0 to 32298
Data columns (total 2 columns):
 #   Column       Non-Null Count  Dtype
---  ------       --------------  -----
 0   feature_idx  32299 non-null  int64
 1   shard        32299 non-null  int64
dtypes: int64(2)
memory usage: 504.8 KB
None

Feature 22219 exists: True
